# Ascon Cryptographic Core: Hardware Testing Dashboard
**Target Board:** ChipWhisperer CW305 (Artix-7 FPGA)
**Algorithm:** Ascon (Custom Vitis HLS Implementation)

This notebook connects to the physical CW305 board over USB, flashes our custom synthesized bitstream (`cw305_top.bit`), configures the onboard PLL clocks, and sends 128-bit test vectors to verify the hardware encryption.

In [ ]:
import chipwhisperer as cw
import time
import binascii

print("Libraries loaded successfully. Ready to connect.")

### Phase 1: Hardware Connection & FPGA Configuration
We connect to the USB interface and push the `.bit` file into the FPGA. **Note:** The `cw305_top.bit` file must be in the exact same folder as this Jupyter Notebook!

In [ ]:
print("Connecting to CW305 Board...")
try:
    # Connect to the board over USB
    target = cw.target(None, cw.targets.CW305)
    print("✅ USB Connection Established!")
    
    print("Flashing Ascon Core Bitstream (This takes a few seconds)...")
    # Flash the FPGA with your custom hardware
    target.fpga.FPGAProgram(open("cw305_top.bit", "rb").read())
    print("✅ FPGA Successfully Flashed with Ascon Core!")
except Exception as e:
    print("❌ ERROR: Could not connect or flash. Is the board plugged in and the driver installed?")
    print(e)

### Phase 2: Core Voltage & PLL Clock Synchronization
Our Ascon core requires a stable clock to process the memory interfaces byte-by-byte. We set the FPGA core voltage to the standard 1.0V and spin up the motherboard's Phase-Locked Loop (PLL) to generate a clean 10.00 MHz clock (ideal for side-channel power analysis).

In [ ]:
print("Configuring Power and Clocks...")

# Set core voltage to 1.0V
target.vccint_set(1.0)

# Enable the PLL and set it to 10.00 MHz
target.pll.pll_enable_set(True)
target.pll.pll_outenable_set(True, 1)
target.pll.pll_outfreq_set(10E6, 1)

# Give the hardware 100 milliseconds to let the clock stabilize
time.sleep(0.1)

# Verify the clock speed
actual_freq = target.pll.pll_outfreq_set(10E6, 1)
print(f"✅ Clocks stable! Target: 10.0 MHz | Actual: {actual_freq / 1E6:.2f} MHz")

### Phase 3: The Encryption Test (Data Bridging)
We define a 16-byte (128-bit) Secret Key and Plaintext. Using the native CW305 wrapper memory addresses, we push this data across the USB cable directly into our Verilog multiplexer bridges. Finally, we pull the hardware trigger (`target.go()`) to start the Ascon engine.

In [ ]:
# 1. Define your 16-byte (128-bit) test vectors
# You can change these bytes later to test different inputs!
secret_key = bytearray([0x00, 0x01, 0x02, 0x03, 0x04, 0x05, 0x06, 0x07, 
                        0x08, 0x09, 0x0A, 0x0B, 0x0C, 0x0D, 0x0E, 0x0F])

plaintext = bytearray([0x48, 0x45, 0x4C, 0x4C, 0x4F, 0x20, 0x41, 0x53, 
                       0x43, 0x4F, 0x4E, 0x20, 0x31, 0x32, 0x33, 0x21]) # "HELLO ASCON 123!"

print("Pushing Key and Plaintext to FPGA Registers...")
# 2. Write data over USB to the hardware registers
target.fpga_write(target.REG_CRYPT_KEY, secret_key)
target.fpga_write(target.REG_CRYPT_TEXTIN, plaintext)

print("Pulling the Hardware Trigger...")
# 3. Pull the Trigger! (Sets the ap_start pin high)
target.go()

# 4. Read the 16-byte Ciphertext back out of the FPGA
ciphertext = target.fpga_read(target.REG_CRYPT_CIPHEROUT, 16)

print("\n" + "="*40)
print("🚀 ASCON ENCRYPTION COMPLETE 🚀")
print("="*40)
print(f"Plaintext (Hex):  {binascii.hexlify(plaintext).decode('utf-8').upper()}")
print(f"Secret Key (Hex): {binascii.hexlify(secret_key).decode('utf-8').upper()}")
print(f"Ciphertext (Hex): {binascii.hexlify(bytearray(ciphertext)).decode('utf-8').upper()}")
print("="*40)

### Phase 4: Safe Disconnect
Always disconnect the target when finished. If you don't run this, the USB port may lock up and require you to physically unplug the board before running the script again.

In [1]:
# Disconnect and free up the USB port
target.dis()
print("✅ Target safely disconnected.")

NameError: name 'target' is not defined